In [1]:
from core.settings import settings
from core.clients import apify_client, apify_tiktok_actor, apify_client_sync, apify_tiktok_actor_sync
from apps.tiktok.models import Video, VideoStats, RawData
from core.database import SessionFactory, SyncSessionFactory, engine
from sqlalchemy import select, func, cast, Date, over
from sqlalchemy.orm import aliased
from sqlalchemy.dialects import postgresql
from datetime import datetime, timedelta, date
from sqlalchemy.exc import IntegrityError
import pandas as pd
from zoneinfo import ZoneInfo
import seaborn as sns
from apps.tiktok.utils import extract_tiktok_username
import matplotlib.pyplot as plt
import random

In [2]:
session = SessionFactory()

In [3]:
date_expr = func.timezone("America/Lima", VideoStats.created_at).cast(Date)
rn_expr = func.row_number().over(
    partition_by=(Video.id, date_expr), order_by=VideoStats.created_at.desc()
)
subquery = (
    select(Video, VideoStats, rn_expr.label("rn"))
    .join(VideoStats, onclause=Video.id == VideoStats.video_id)
    .subquery()
)

v_alias = aliased(Video, subquery)
vs_alias = aliased(VideoStats, subquery)

stmt = (
    select(v_alias, vs_alias)
    .select_from(subquery)
    .where(subquery.c.rn == 1)
    .order_by(vs_alias.created_at.asc())
)
results = (await session.execute(stmt)).all()

data = [
    {
        "username": extract_tiktok_username(video.url),
        "url": video.url,
        "date_of_stats": stats.created_at.astimezone(ZoneInfo("America/Lima")).date(),
        "diggs": stats.diggs,
        "shares": stats.shares,
        "plays": stats.plays,
        "collects": stats.collects,
        "comments": stats.comments,
        "reposts": stats.reposts,
    }
    for video, stats in results
]
data

[{'username': 'mamiofertass',
  'url': 'https://www.tiktok.com/@mamiofertass/video/7622025788581334279',
  'date_of_stats': datetime.date(2026, 3, 31),
  'diggs': 91,
  'shares': 12,
  'plays': 1247,
  'collects': 7,
  'comments': 52,
  'reposts': 0},
 {'username': 'andreody_coach',
  'url': 'https://www.tiktok.com/@andreody_coach/video/7623189687041117447',
  'date_of_stats': datetime.date(2026, 3, 31),
  'diggs': 148,
  'shares': 16,
  'plays': 4851,
  'collects': 40,
  'comments': 5,
  'reposts': 0},
 {'username': 'andreody_coach',
  'url': 'https://www.tiktok.com/@andreody_coach/video/7622066906131991815',
  'date_of_stats': datetime.date(2026, 3, 31),
  'diggs': 164,
  'shares': 6,
  'plays': 5504,
  'collects': 38,
  'comments': 3,
  'reposts': 0},
 {'username': 'brendagonzalesr',
  'url': 'https://www.tiktok.com/@brendagonzalesr/video/7621945617077619989',
  'date_of_stats': datetime.date(2026, 3, 31),
  'diggs': 53,
  'shares': 37,
  'plays': 1591,
  'collects': 3,
  'comments'

In [5]:
START_DATE = date(2026, 3, 31)

for video, stats in results:
    if stats.created_at.astimezone(ZoneInfo("America/Lima")).date() == date(2026, 3, 31):
        print(video.url, stats.created_at, stats.plays)
        for i in range(1, 5):
            video_stats = VideoStats(
                video=video,
                diggs=int(stats.diggs*(random.randint(65, 75)/100)**i),
                shares=int(stats.shares*(random.randint(65, 75)/100)**i),
                plays=int(stats.plays*(random.randint(65, 75)/100)**i),
                collects=int(stats.collects*(random.randint(65, 75)/100)**i),
                comments=int(stats.comments*(random.randint(65, 75)/100)**i),
                reposts=0,
                created_at=datetime.combine(START_DATE - timedelta(days=i), datetime.max.time(), ZoneInfo("America/Lima"))
            )
            session.add(video_stats)
await session.commit()

https://www.tiktok.com/@mamiofertass/video/7622025788581334279 2026-03-31 20:50:00.050262-04:00 1247
https://www.tiktok.com/@andreody_coach/video/7623189687041117447 2026-03-31 20:50:00.054275-04:00 4851
https://www.tiktok.com/@andreody_coach/video/7622066906131991815 2026-03-31 20:50:00.057265-04:00 5504
https://www.tiktok.com/@brendagonzalesr/video/7621945617077619989 2026-03-31 20:50:00.060271-04:00 1591
https://www.tiktok.com/@pauespinozao/video/7623206456636099860 2026-03-31 20:50:00.062273-04:00 353
https://www.tiktok.com/@mamiofertass/video/7623133210653838613 2026-03-31 20:50:00.066277-04:00 583
https://www.tiktok.com/@brendagonzalesr/video/7623059137261227284 2026-03-31 20:50:00.069279-04:00 1167
https://www.tiktok.com/@kairycc_/video/7623203883661184277 2026-03-31 20:50:00.071266-04:00 2343
https://www.tiktok.com/@sumalavia.7/video/7623203343002766610 2026-03-31 20:50:00.074428-04:00 471
https://www.tiktok.com/@sumalavia.7/video/7622090562602028306 2026-03-31 20:50:00.076510-

In [33]:
START_DATE = date(2026, 3, 27)
datetime.combine(START_DATE, datetime.min.time())

datetime.datetime(2026, 3, 27, 0, 0)

In [111]:
video.url

'https://www.tiktok.com/@mamiofertass/video/7623133210653838613'

,username,url,date_of_stats,diggs,shares,plays,collects,comments,reposts
0,mamiofertass,https://www.tiktok.com/@mamiofertass/video/762...,2026-03-31,91,12,1247,7,52,0
1,andreody_coach,https://www.tiktok.com/@andreody_coach/video/7...,2026-03-31,148,16,4851,40,5,0
2,andreody_coach,https://www.tiktok.com/@andreody_coach/video/7...,2026-03-31,164,6,5504,38,3,0
3,brendagonzalesr,https://www.tiktok.com/@brendagonzalesr/video/...,2026-03-31,53,37,1591,3,9,0
4,pauespinozao,https://www.tiktok.com/@pauespinozao/video/762...,2026-03-31,69,12,353,5,34,0
5,mamiofertass,https://www.tiktok.com/@mamiofertass/video/762...,2026-03-31,67,12,583,3,49,0
6,brendagonzalesr,https://www.tiktok.com/@brendagonzalesr/video/...,2026-03-31,49,4,1167,9,4,0
7,kairycc_,https://www.tiktok.com/@kairycc_/video/7623203...,2026-03-31,258,19,2343,16,13,0
8,sumalavia.7,https://www.tiktok.com/@sumalavia.7/video/7623...,2026-03-31,42,28,471,4,25,0
9,sumalavia.7,https://www.tiktok.com/@sumalavia.7/video/7622...,2026-03-31,50,40,528,4,26,0


In [ ]:
stmt = select(Video)

In [10]:
session = SyncSessionFactory()
session

In [18]:
videos_urls = """https://www.tiktok.com/@sumalavia.7/video/7622090562602028306
https://www.tiktok.com/@andreody_coach/video/7622066906131991815
https://www.tiktok.com/@brendagonzalesr/video/7621945617077619989
https://www.tiktok.com/@mamiofertass/video/7622025788581334279
https://www.tiktok.com/@pauespinozao/video/7623206456636099860
https://www.tiktok.com/@kairycc_/video/7623203883661184277
https://www.tiktok.com/@sumalavia.7/video/7623203343002766610
https://www.tiktok.com/@andreody_coach/video/7623189687041117447
https://www.tiktok.com/@brendagonzalesr/video/7623059137261227284
https://www.tiktok.com/@mamiofertass/video/7623133210653838613"""
for url in videos_urls.splitlines():
    try:
        video = Video(url=url)
        session.add(video)
        await session.commit()
    except IntegrityError:
        await session.rollback()

In [11]:
stmt = select(Video)
videos = session.scalars(stmt).all()
videos

[Video(id=1, url='https://www.tiktok.com/@sumalavia.7/video/7622090562602028306', last_stat_id=None),
 Video(id=3, url='https://www.tiktok.com/@brendagonzalesr/video/7621945617077619989', last_stat_id=None),
 Video(id=4, url='https://www.tiktok.com/@andreody_coach/video/7622066906131991815', last_stat_id=None),
 Video(id=5, url='https://www.tiktok.com/@mamiofertass/video/7622025788581334279', last_stat_id=None),
 Video(id=6, url='https://www.tiktok.com/@mamiofertass/video/7623133210653838613', last_stat_id=None),
 Video(id=7, url='https://www.tiktok.com/@brendagonzalesr/video/7623059137261227284', last_stat_id=None),
 Video(id=16, url='https://www.tiktok.com/@pauespinozao/video/7623206456636099860', last_stat_id=None),
 Video(id=17, url='https://www.tiktok.com/@kairycc_/video/7623203883661184277', last_stat_id=None),
 Video(id=18, url='https://www.tiktok.com/@sumalavia.7/video/7623203343002766610', last_stat_id=None),
 Video(id=19, url='https://www.tiktok.com/@andreody_coach/video/7623

In [14]:
run_input = {
  "postURLs": [video.url for video in videos],
}
run = apify_tiktok_actor_sync.call(run_input=run_input)

[apify.tiktok-scraper runId:Yr8eWofj0QoQarlo6] -> Status: RUNNING, Message: 
[apify.tiktok-scraper runId:Yr8eWofj0QoQarlo6] -> 2026-04-01T17:50:24.467Z ACTOR: Pulling container image of build hzCuJPLnJcagnd9O0 from registry.
[apify.tiktok-scraper runId:Yr8eWofj0QoQarlo6] -> 2026-04-01T17:50:24.469Z ACTOR: Creating container.
[apify.tiktok-scraper runId:Yr8eWofj0QoQarlo6] -> 2026-04-01T17:50:24.531Z ACTOR: Starting container.
[apify.tiktok-scraper runId:Yr8eWofj0QoQarlo6] -> 2026-04-01T17:50:24.532Z ACTOR: Running under "LIMITED_PERMISSIONS" permission level.
[apify.tiktok-scraper runId:Yr8eWofj0QoQarlo6] -> 2026-04-01T17:50:24.838Z Running on architecture: x86_64
[apify.tiktok-scraper runId:Yr8eWofj0QoQarlo6] -> 2026-04-01T17:50:24.839Z Will run command: xvfb-run -a -s "-ac -screen 0 1920x1080x24+32 -nolisten tcp" /bin/bash -o pipefail -c bash -c "    java         --enable-native-access=ALL-UNNAMED         -cp classes-test:classes-main:lib/*         com.zhiliaoapp.musically.MetaSec    

In [21]:
async for item in apify_client.dataset(run["defaultDatasetId"]).iterate_items():
    url = item["webVideoUrl"]
    video = next(video for video in videos if video.url == item["webVideoUrl"])

    raw_data = RawData(data=item)
    session.add(raw_data)

    video_stats = VideoStats(
        video=video,
        raw_data=raw_data,
        diggs=item["diggCount"],
        shares=item["shareCount"],
        plays=item["playCount"],
        collects=item["collectCount"],
        comments=item["commentCount"],
        reposts=item["repostCount"],
    )
    session.add(video_stats)
    await session.commit()

In [17]:
import schedule
import time as time_module
from sqlalchemy import select
from core.database import SyncSessionFactory, load_models
from apps.tiktok.models import Video, RawData, VideoStats
from core.clients import apify_tiktok_actor_sync, apify_client_sync

load_models()

def scrape_tiktok():
    with SyncSessionFactory() as session:
        stmt = select(Video)
        videos = session.scalars(stmt).all()
        run_input = {
            "postURLs": [video.url for video in videos]
        }
        run = apify_tiktok_actor_sync.call(run_input=run_input)
        for item in apify_client_sync.dataset(run["defaultDatasetId"]).iterate_items():
            url = item["webVideoUrl"]
            video = next(video for video in videos if video.url == item["webVideoUrl"])
            raw_data = RawData(data=item)
            session.add(raw_data)

            video_stats = VideoStats(
                video=video,
                raw_data=raw_data,
                diggs=item["diggCount"],
                shares=item["shareCount"],
                plays=item["playCount"],
                collects=item["collectCount"],
                comments=item["commentCount"],
                reposts=item["repostCount"],
            )

            video.last_stat = video_stats

            session.add(video_stats)
            session.commit()

scrape_tiktok()

[apify.tiktok-scraper runId:TOa4r9PesNdMTv2We] -> Status: RUNNING, Message: 
[apify.tiktok-scraper runId:TOa4r9PesNdMTv2We] -> 2026-04-01T18:07:46.830Z ACTOR: Pulling container image of build hzCuJPLnJcagnd9O0 from registry.
[apify.tiktok-scraper runId:TOa4r9PesNdMTv2We] -> 2026-04-01T18:07:46.834Z ACTOR: Creating container.
[apify.tiktok-scraper runId:TOa4r9PesNdMTv2We] -> 2026-04-01T18:07:47.224Z ACTOR: Starting container.
[apify.tiktok-scraper runId:TOa4r9PesNdMTv2We] -> 2026-04-01T18:07:47.225Z ACTOR: Running under "LIMITED_PERMISSIONS" permission level.
[apify.tiktok-scraper runId:TOa4r9PesNdMTv2We] -> 2026-04-01T18:07:47.609Z Running on architecture: x86_64
[apify.tiktok-scraper runId:TOa4r9PesNdMTv2We] -> 2026-04-01T18:07:47.613Z Will run command: xvfb-run -a -s "-ac -screen 0 1920x1080x24+32 -nolisten tcp" /bin/bash -o pipefail -c bash -c "    java         --enable-native-access=ALL-UNNAMED         -cp classes-test:classes-main:lib/*         com.zhiliaoapp.musically.MetaSec    

In [16]:
video_stats = VideoStats(
    video=videos[0],
    raw_data=raw_data,
    diggs=item["diggCount"],
    shares=item["shareCount"],
    plays=item["playCount"],
    collects=item["collectCount"],
    comments=item["commentCount"],
    reposts=item["repostCount"],
)
session.add(video_stats)
await session.commit()

sys:1: SAWarning: Object of type <VideoStats> not in session, add operation along 'Video.stats' will not proceed


In [19]:
list(session)

[Video(id=1, url='https://www.tiktok.com/@sumalavia.7/video/7622090562602028306', last_stat_id=None),
 RawData(id=1, data={'id': '7622090562602028306', 'text': '#Publicidad \n\nSé que el ceviche te queda rico, pero con el Paiche de Yummei te quedará aún mejor. 🐟✨\n\n\n\nDisponible en @wongoficial , aporta una frescura y textura firme que marcan la diferencia. 🩵\n\n\n\nMantén la tradición en casa con calidad excepcional. 🛒\n\n\n\n@Blooper Strategy Partner \n\n#SemanaSanta #Ceviche #receta #creatorsearchinsights ', 'textLanguage': 'es', 'createTime': 1774656260, 'createTimeISO': '2026-03-28T00:04:20.000Z', 'locationCreated': 'PE', 'isAd': False, 'authorMeta': {'id': '7004858351788295174', 'name': 'sumalavia.7', 'profileUrl': 'https://www.tiktok.com/@sumalavia.7', 'nickName': 'Alberto Sumalavia', 'verified': False, 'signature': '🧑\u200d🍳 Creador gastronómico | Cocinero 🇵🇪', 'avatar': 'https://p16-common-sign.tiktokcdn-us.com/tos-alisg-avt-0068/0141375173e071ffe61bc4c696e94579~tplv-tiktokx